# 4 · Agente

**Módulo 1 · Introducción.** El proyecto del módulo es un agente de cálculo simbólico. El router de la lección 3 deja decidir al modelo una sola vez y termina con el resultado de la herramienta, que el modelo nunca lee. Esta lección cambia una arista, $\texttt{tools} \to \texttt{END}$ por $\texttt{tools} \to \texttt{modelo}$: el modelo lee cada resultado y vuelve a decidir, hasta que responde sin pedidos. Ese grafo es el agente.

La arista nueva cierra un ciclo, y con el ciclo vuelve la pregunta de la lección 1: ¿la ejecución termina? Se define el agente, se cuentan sus supersteps, se prueba que ninguna variante garantiza que termine y se le agrega una guardia con `RemainingSteps`, que lo hace llegar a `END` por sí mismo, sin `GraphRecursionError`.

**Dónde corre.** Las secciones 1 a 5 y la profundización no usan el modelo y corren en el PC, con modelos guionados. La sección 6 usa `qwen3:8b`, que corre en Kaggle con una GPU; el [índice del módulo](README.md) explica cómo abrir el notebook allí.

Requisito: `pip install "langgraph>=1.2" "langchain-ollama>=1.1" "sympy>=1.14"`. Las salidas de este notebook son de langgraph 1.2.17, langgraph-prebuilt 1.1.1, langchain-core 1.7.0, langchain-ollama 1.1.0, sympy 1.14.0 y Ollama 0.40.13 con `qwen3:8b`, en una GPU T4 de Kaggle.

**Preparación.** En Kaggle, la celda siguiente instala las dependencias y Ollama, arranca el servidor y descarga `qwen3:8b`, en poco más de un minuto. En el PC no hace nada.

In [ ]:
import os
import subprocess
import time
import urllib.request

EN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ


def correr(comando: str) -> None:
    """Corre un comando de shell y, si falla, muestra el final de su error."""
    r = subprocess.run(comando, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"falló: {comando}\n{r.stderr[-2000:]}")


def preparar_kaggle() -> None:
    """Instala las dependencias y Ollama, arranca el servidor y descarga qwen3:8b."""
    correr("pip install -q 'langgraph==1.2.17' 'langgraph-prebuilt==1.1.1' 'langchain-core==1.7.0'"
           " 'langchain-ollama==1.1.0' 'sympy==1.14.0'")
    correr("(command -v zstd || (apt-get -qq update && apt-get -qq install -y zstd))"
           " && curl -fsSL https://ollama.com/install.sh | sh")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/version")
            break
        except OSError:
            time.sleep(1)
    else:
        raise RuntimeError("el servidor de Ollama no respondió en 60 s")
    correr("ollama pull qwen3:8b")


if EN_KAGGLE:
    preparar_kaggle()

In [ ]:
from importlib.metadata import version

for paquete in ("langgraph", "langgraph-prebuilt", "langchain-core", "langchain-ollama", "sympy"):
    print(paquete, version(paquete))

## 1. El problema

Por la Proposición 3.5, en el router el nodo del modelo corre una sola vez, antes de que exista cualquier respuesta de herramienta. Un pedido como «Deriva x³ y después integra el resultado» necesita dos decisiones, y la segunda depende del resultado de la primera: para pedir una primitiva de $3x^2$, el modelo tiene que haber leído $3x^2$. En la lección 3, `qwen3:8b` escribió los dos pedidos a la vez, los dos con $x^3$, y el router terminó en $x^4/4$.

Falta volver al modelo después de `tools`. Basta cambiar la arista fija $\texttt{tools} \to \texttt{END}$ del router por $\texttt{tools} \to \texttt{modelo}$. El grafo queda con un ciclo, $\texttt{modelo} \to \texttt{tools} \to \texttt{modelo}$, del que la ejecución sale por la arista condicional cuando el modelo responde sin pedidos.

**Mapa de la lección.** El agente con «Deriva x³ y después integra el resultado», un superstep a la vez. El grafo se define en la sección 2; la 3 cuenta sus supersteps, la 4 muestra que puede no terminar y la 5 le agrega la guardia que lo hace terminar. La sección 6 lo corre con el modelo.

<img src="../../imagenes/diagramas/modulo-1/agente.svg" alt="El agente: START lleva a modelo; de modelo, una arista condicional va a tools si el último mensaje trae pedidos y a END si no; tools vuelve a modelo. Con «Deriva x³ y después integra el resultado», el modelo pide derivar(x**3), tools escribe 3*x**2, el modelo lo lee y pide integrar(3*x**2), tools escribe x**3 y el modelo responde con texto, así que la ejecución va a END en el superstep 5" width="560">

Las herramientas, `linea` y el nodo `tools_seguro` son los de la lección 3:

In [ ]:
from sympy import Symbol, diff, integrate, solve
from sympy.parsing.sympy_parser import convert_xor, parse_expr, standard_transformations

TRANSFORMACIONES = standard_transformations + (convert_xor,)


def leer(texto: str):
    """Convierte texto en sintaxis de SymPy en una expresión; «^» vale como potencia."""
    return parse_expr(texto, transformations=TRANSFORMACIONES)


def derivar(expresion: str, variable: str = "x") -> str:
    """Deriva una expresión respecto de una variable.

    Args:
        expresion: expresión en sintaxis de SymPy, por ejemplo x**2*sin(x).
        variable: variable respecto de la que se deriva.
    """
    return str(diff(leer(expresion), Symbol(variable)))


def integrar(expresion: str, variable: str = "x") -> str:
    """Calcula una primitiva de una expresión, sin constante de integración.

    Args:
        expresion: expresión en sintaxis de SymPy, por ejemplo 1/(1 + x**2).
        variable: variable de integración.
    """
    return str(integrate(leer(expresion), Symbol(variable)))


def resolver(ecuacion: str, variable: str = "x") -> str:
    """Resuelve una ecuación en una incógnita y devuelve la lista de soluciones.

    Args:
        ecuacion: ecuación en sintaxis de SymPy, por ejemplo x**2 - 2 = 0. Sin signo =, se iguala a cero.
        variable: incógnita.
    """
    izquierda, _, derecha = ecuacion.replace("==", "=").partition("=")  # «==», como en Python, vale como «=»
    return str(solve(leer(izquierda) - leer(derecha or "0"), Symbol(variable)))


HERRAMIENTAS = [derivar, integrar, resolver]

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.messages.tool import tool_call
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

tools_seguro = ToolNode(HERRAMIENTAS, handle_tool_errors=True)


def linea(m) -> str:
    """Rol y contenido de un mensaje; sus pedidos si es de rol ai, y su γ y estado si es de rol tool."""
    if m.type == "ai" and m.tool_calls:
        return "ai    pide " + ", ".join(f"{p['name']}({p['args']}) γ={p['id'][:8]}" for p in m.tool_calls)
    if m.type == "tool":
        return f"tool  {m.content!r}  γ={m.tool_call_id[:8]}  {m.status}"
    return f"{m.type:5} {m.content!r}"

## 2. El agente

**Definición 4.1 (agente).** Sean $\varphi_M$ el nodo de un modelo (lección 2, §6), $\varphi_H$ el nodo de herramientas con la respuesta de la Definición 3.2 y $c$ la arista `tools_condition` (Definición 3.3). El agente es el grafo con los nodos `modelo` y `tools`, de funciones $\varphi_M$ y $\varphi_H$, y tres aristas: la fija $\texttt{START} \to \texttt{modelo}$, la condicional $c$ desde `modelo` y la fija $\texttt{tools} \to \texttt{modelo}$.

Difiere del router en una sola arista. Desde `tools` ya no se llega a `END`: la única salida es que $c$ elija `END`, y eso ocurre cuando el último mensaje, que acaba de escribir el modelo, no trae pedidos. `construir_agente` recibe la función del nodo del modelo y, desde la sección 5, el esquema del estado.

In [ ]:
def construir_agente(modelo, tools: ToolNode, esquema=MessagesState):
    """START → modelo; de modelo, a tools o a END según tools_condition; tools → modelo."""
    constructor = StateGraph(esquema)
    constructor.add_node("modelo", modelo)
    constructor.add_node("tools", tools)
    constructor.add_edge(START, "modelo")
    constructor.add_conditional_edges("modelo", tools_condition)
    constructor.add_edge("tools", "modelo")
    return constructor.compile()

Para probar el agente sin GPU hace falta un modelo guionado que no escriba siempre lo mismo: el de la lección 3 pediría la misma herramienta en cada vuelta.

**Definición 4.2 (modelo guionado por turnos).** Sea $(m_1, \dots, m_q)$ una sucesión de mensajes de rol ai, y sea $a(c)$ la cantidad de mensajes de rol ai de una conversación $c$. El modelo guionado por turnos escribe $m_{a(c)+1}$ si $a(c) < q$, y $m_q$ si no. Como núcleo, $M(\cdot \mid c) = \delta_{m_j}$ con $j = \min(a(c) + 1, q)$.

Con $q = 1$ es el modelo guionado de la Definición 3.4. En el agente, cada vez que el modelo corre encuentra un mensaje de rol ai más, así que el guion se lee en orden. El de abajo resuelve «Deriva x³ y después integra el resultado» en dos rondas, como debería hacerlo el modelo.

In [ ]:
from IPython.display import Image, display


def por_turnos(*guion: AIMessage):
    """El nodo de un modelo guionado por turnos: si la conversación tiene i mensajes de rol ai, escribe guion[i]."""

    def modelo(s: MessagesState) -> dict:
        i = sum(1 for x in s["messages"] if x.type == "ai")
        return {"messages": [guion[min(i, len(guion) - 1)]]}

    return modelo


guion = (
    AIMessage("", id="m1", tool_calls=[tool_call(name="derivar", args={"expresion": "x**3"}, id="g1")]),
    AIMessage("", id="m2", tool_calls=[tool_call(name="integrar", args={"expresion": "3*x**2"}, id="g2")]),
    AIMessage("La derivada es 3*x**2, y una primitiva de 3*x**2 es x**3.", id="m3"),
)
agente = construir_agente(por_turnos(*guion), tools_seguro)
display(Image(agente.get_graph().draw_mermaid_png()))

La arista $\texttt{tools} \to \texttt{modelo}$ es continua porque es fija; las punteadas son las dos salidas de $c$. A diferencia del router, `tools` no tiene arista a `END`.

## 3. Cuántos supersteps usa

**Proposición 4.3 (ejecución del agente).** Sea $e \in C$ una entrada cuyos mensajes tienen identificador. Supóngase que, en sus llamadas sucesivas, el nodo del modelo escribe $m_1, m_2, \dots$, con identificadores distintos entre sí y que no están en $\iota(e)$; que $m_1, \dots, m_r$ traen $k_1, \dots, k_r \ge 1$ pedidos, y que $m_{r+1}$ no trae pedidos. Sea $T_i = (t_{i,1}, \dots, t_{i,k_i})$ la sucesión de respuestas a los pedidos de $m_i$ (Definición 3.2). Con `handle_tool_errors=True` y un `recursion_limit` de al menos $2r + 2$:

1. `modelo` corre en los supersteps impares $1, 3, \dots, 2r + 1$ y `tools` en los pares $2, 4, \dots, 2r$, así que la ejecución usa $2r + 2$ supersteps, contando el de `START`;
2. la conversación final es la de $e$ seguida de $m_1$, $T_1$, $m_2$, $T_2$, …, $m_r$, $T_r$ y $m_{r+1}$;
3. en el superstep $2i + 1$, el modelo lee una conversación que contiene $T_1, \dots, T_i$.

*Demostración.* Se prueba por inducción en $i = 1, \dots, r + 1$ que en el superstep $2i - 1$ corre solo `modelo` y que, al cerrarlo, la conversación es la de $e$ seguida de $m_1, T_1, \dots, m_{i-1}, T_{i-1}$ y $m_i$.

Para $i = 1$, el argumento es el de la Proposición 3.5: `START` escribe $e$ en el superstep 0 y `modelo` escribe $(m_1)$ en el 1, que se agrega al final porque su identificador no está en $\iota(e)$.

Supóngase cierto para un $i \le r$. Como $m_i$ trae pedidos, $c$ elige `tools`, y en el superstep $2i$ corre solo `tools`. Busca el último mensaje de rol ai, que es $m_i$, y escribe $T_i$; las respuestas reciben identificadores nuevos, así que $\oplus$ las agrega al final. La arista fija $\texttt{tools} \to \texttt{modelo}$ activa `modelo` en el superstep $2i + 1$, que lee esa conversación, con $T_1, \dots, T_i$, y escribe $m_{i+1}$, que también va al final. Eso prueba el paso y el punto 3.

Para $i = r + 1$, $m_{r+1}$ no trae pedidos y $c$ elige `END`, así que en el superstep $2r + 2$ no hay nodos activos. El último superstep es el $2r + 1$; con el de `START` son $2r + 2$, que `recursion_limit` permite. $\square$

Dos consecuencias. `tools` nunca corre dos veces seguidas: la única arista que entra a `tools` sale de `modelo`, y la única que sale de `tools` vuelve a `modelo`, así que entre dos rondas de herramientas siempre hay una llamada al modelo. Y la cuenta depende de las rondas, no de los pedidos: `tools` responde los $k_i$ pedidos de $m_i$ en un solo superstep (Profundización 3.A).

In [ ]:
entrada = {"messages": [HumanMessage("Deriva x³ y después integra el resultado", id="h1")]}

print("después de START corren", [list(u) for u in agente.stream(entrada, stream_mode="updates")])
for x in agente.invoke(entrada)["messages"]:
    print(linea(x))

Con $r = 2$ son seis supersteps: `START`, modelo, tools, modelo, tools, modelo. En el superstep 3, el modelo lee `3*x**2` antes de pedir `integrar`, que es lo que el router no podía hacer.

**Corolario 4.4.** Con las hipótesis de la Proposición 4.3, el menor `recursion_limit` con que la ejecución llega a `END` es $2r + 2$.

*Demostración.* Con $2r + 2$ llega, por la Proposición 4.3. Con un límite menor, la ejecución necesita más supersteps de los que el límite permite, y `GraphRecursionError` la detiene (Definición 1.15). $\square$

In [ ]:
from langgraph.errors import GraphRecursionError

for limite in (5, 6):
    try:
        agente.invoke(entrada, {"recursion_limit": limite})
        print(f"recursion_limit = {limite}: llega a END")
    except GraphRecursionError:
        print(f"recursion_limit = {limite}: GraphRecursionError")

## 4. ¿Termina?

En Euclides, el valor de $b$ era una variante (Proposición 1.17): un natural que baja en cada superstep. En el agente, lo que cambia en cada superstep es el largo de la conversación, y cambia hacia arriba: `modelo` agrega un mensaje y `tools`, $k_i \ge 1$. Que algo crezca no prueba nada. Una variante sirve porque baja en $\mathbb{N}$, y en $\mathbb{N}$ no se puede bajar para siempre; un largo que sube sí puede subir para siempre. Basta un modelo que pida herramientas en cada mensaje.

**Definición 4.5 (modelo insistente).** El modelo insistente escribe, con cualquier conversación, un mensaje de rol ai con un pedido, `derivar(x**3)`. El mensaje y el pedido llevan identificadores nuevos en cada llamada.

Antes de ejecutarlo, un detalle de cómo corta `recursion_limit`.

**Observación (el superstep $L$).** Sea $L$ el `recursion_limit`. En langgraph 1.2.17, el límite se revisa al comienzo de cada superstep (método `tick`, en `langgraph/pregel/_loop.py`), y la ejecución se detiene con `GraphRecursionError` en el primer superstep $t > L$, aunque ya no queden nodos activos. Los nodos activos en el superstep $L$ alcanzan a correr. Así, una ejecución llega a `END` si y solo si su último superstep con nodos activos es a lo más el $L - 1$, como dice la Definición 1.15, y cuando no llega, los nodos del superstep $L$ ya corrieron. Si uno de ellos es el modelo, es una llamada que no sirve.

**Proposición 4.6 (no hay variante).** Con el modelo insistente, la ejecución del agente no llega a `END` con ningún `recursion_limit` $L$: se detiene con `GraphRecursionError` después de $\lceil L/2 \rceil$ llamadas al modelo. En consecuencia, ninguna función $V : S \to \mathbb{N}$ es una variante del agente para todo modelo.

*Demostración.* Cada mensaje del modelo insistente trae un pedido, así que $c$ elige siempre `tools`, y `tools` vuelve siempre a `modelo`: hay un nodo activo en todo superstep $t \ge 1$. Por la observación, corren los supersteps $1, \dots, L$ y la ejecución se detiene con `GraphRecursionError`. `modelo` corre en los impares de $\{1, \dots, L\}$, que son $\lceil L/2 \rceil$.

Si $V$ fuera una variante con este modelo, bajaría en cada superstep en que corre un nodo, y en $\mathbb{N}$ eso ocurre a lo más $V(s_0)$ veces. Con $L \ge V(s_0) + 2$, el último superstep con nodos activos sería a lo más el $V(s_0) \le L - 2$, y la ejecución llegaría a `END`. $\square$

In [ ]:
import uuid


def insistente(s: MessagesState) -> dict:
    """El nodo del modelo insistente: pide derivar(x**3) con cualquier conversación."""
    pedido = tool_call(name="derivar", args={"expresion": "x**3"}, id=str(uuid.uuid4()))
    return {"messages": [AIMessage("", id=str(uuid.uuid4()), tool_calls=[pedido])]}


def contar(nodo):
    """Envuelve el nodo de un modelo y cuenta sus llamadas en el atributo `llamadas`."""

    def envuelto(s: MessagesState) -> dict:
        envuelto.llamadas += 1
        return nodo(s)

    envuelto.llamadas = 0
    return envuelto


for limite in (6, 7, 25):
    modelo = contar(insistente)
    try:
        construir_agente(modelo, tools_seguro).invoke(entrada, {"recursion_limit": limite})
    except GraphRecursionError:
        print(f"recursion_limit = {limite}: GraphRecursionError después de {modelo.llamadas} llamadas al modelo")

Con el valor por defecto, $L = 10\,007$, el modelo insistente correría $5\,004$ veces antes del error. Con `qwen3:8b`, que tarda segundos en cada llamada, serían horas de GPU para terminar en un error. Y `recursion_limit` corta la ejecución, pero no la termina: no llega a `END`, e `invoke` no devuelve ningún estado.

## 5. Pasos restantes

En la lección 1, la cota para Newton se puso en la arista (Proposición 1.19). En el agente, la arista es `tools_condition`, que solo mira el último mensaje. La cota va en el nodo del modelo, que es el que decide si se piden herramientas, y para eso necesita saber cuántos supersteps quedan.

**Definición 4.7 (pasos restantes).** Sea $L$ el `recursion_limit` de una ejecución. Los pasos restantes en el superstep $t$ son $\rho_t = L - t$. En LangGraph, una clave del esquema anotada con `RemainingSteps`, de `langgraph.managed`, es un valor administrado: ningún nodo la escribe, y el nodo que corre en el superstep $t$ la lee con valor $\rho_t$ (`RemainingStepsManager`, en `langgraph/managed/is_last_step.py`).

`EstadoAgente` agrega la clave `pasos_restantes` a `MessagesState`. No aparece en el estado que devuelve `invoke`: solo la leen los nodos.

In [ ]:
from langgraph.managed import RemainingSteps


class EstadoAgente(MessagesState):
    pasos_restantes: RemainingSteps


def insistente_que_mira(s: EstadoAgente) -> dict:
    """El modelo insistente, que antes muestra los pasos restantes."""
    print("modelo lee pasos_restantes =", s["pasos_restantes"])
    return insistente(s)


try:
    construir_agente(insistente_que_mira, tools_seguro, EstadoAgente).invoke(entrada, {"recursion_limit": 7})
except GraphRecursionError:
    print("GraphRecursionError")

Con $L = 7$, `modelo` corre en los supersteps 1, 3, 5 y 7 y lee $\rho = 6, 4, 2, 0$. La última llamada es la del superstep $L$, que la observación de la sección 4 anticipaba.

Si el modelo pide herramientas en el superstep $t$, `tools` corre en el $t + 1$ y `modelo` en el $t + 2$. Para llegar a `END`, ese superstep tiene que ser a lo más el $L - 1$, es decir, $\rho_t = L - t \ge 3$. Con menos pasos, pedir herramientas ya no sirve.

**Definición 4.8 (guardia).** Sea $\varphi_M$ el nodo de un modelo. El nodo con guardia toma la muestra $m$ de $\varphi_M$ en el superstep $t$. Si $m$ trae pedidos y $\rho_t < 3$, escribe en su lugar un mensaje de rol ai sin pedidos, con el aviso «No alcancé a terminar: se acabaron los pasos.»; si no, escribe $m$.

**Proposición 4.9 (el agente con guardia termina).** Sea $L \ge 2$ el `recursion_limit`, y supóngase que las herramientas terminan. Con `handle_tool_errors=True` y el nodo con guardia, la ejecución del agente llega a `END` con todo modelo, en a lo más $L$ supersteps y sin `GraphRecursionError`. El modelo corre a lo más $\lfloor L/2 \rfloor$ veces.

*Demostración.* Como en la Proposición 4.3, `modelo` y `tools` se alternan: `modelo` corre en supersteps impares y `tools` en pares. Se prueba que todo superstep en que corre `modelo` cumple $\rho_t \ge 1$. El primero es el 1, con $\rho_1 = L - 1 \ge 1$. Si `modelo` corre en el superstep $t$ y escribe un mensaje sin pedidos, $c$ elige `END`. Si lo escribe con pedidos, la guardia no actuó, así que $\rho_t \ge 3$: `tools` corre en el $t + 1$ y `modelo` en el $t + 2$, con $\rho_{t+2} = \rho_t - 2 \ge 1$.

Entonces todo superstep con nodos activos cumple $t \le L - 1$. Hay finitos, así que la ejecución deja de tener nodos activos, y por la observación de la sección 4 llega a `END` en a lo más $L$ supersteps, sin `GraphRecursionError`. `modelo` corre en impares menores que $L$, que son $\lfloor L/2 \rfloor$. $\square$

Los pasos restantes bajan en 1 en cada superstep, como una variante. Pero los pone LangGraph, no el modelo, y cuando se acaban la ejecución termina en `GraphRecursionError`. La guardia es la que hace salir al agente por `END` antes de que se acaben.

In [ ]:
AVISO = "No alcancé a terminar: se acabaron los pasos."


def con_guardia(modelo, umbral: int = 3):
    """El nodo del modelo con guardia: si quedan menos de `umbral` pasos, cambia un mensaje con pedidos por AVISO."""

    def nodo(s: EstadoAgente) -> dict:
        m = modelo(s)["messages"][-1]
        if m.tool_calls and s["pasos_restantes"] < umbral:
            m = AIMessage(AVISO, id=m.id)
        return {"messages": [m]}

    return nodo


for limite in range(2, 10):
    modelo = contar(insistente)
    final = construir_agente(con_guardia(modelo), tools_seguro, EstadoAgente).invoke(entrada, {"recursion_limit": limite})
    print(f"recursion_limit = {limite}: llega a END; llamadas al modelo = {modelo.llamadas}")
print("último mensaje:", repr(final["messages"][-1].content))

El modelo insistente pide herramientas en todos sus mensajes, y aun así el agente llega a `END` con todo límite, después de $\lfloor L/2 \rfloor$ llamadas y con el aviso al final. La guardia, además, solo actúa en ejecuciones que sin ella terminarían en `GraphRecursionError`.

**Proposición 4.10 (la guardia no cambia lo que terminaba).** Fijada la sucesión de mensajes $m_1, m_2, \dots$ que escribe el modelo, si la ejecución sin guardia llega a `END` con `recursion_limit` $L$, la ejecución con guardia escribe los mismos mensajes y termina en el mismo estado.

*Demostración.* Por la Proposición 4.3, sin guardia hay $r$ rondas, con $2r + 2 \le L$. El mensaje $m_i$, con $i \le r$, se escribe en el superstep $2i - 1$, con $\rho = L - 2i + 1 \ge 2(r - i) + 3 \ge 3$. La guardia no cambia ninguno de esos mensajes, y $m_{r+1}$ no trae pedidos, así que tampoco lo cambia. $\square$

Con el guion de la sección 3, que necesita $L = 6$:

In [ ]:
for limite in (5, 6):
    agente_con_guardia = construir_agente(con_guardia(por_turnos(*guion)), tools_seguro, EstadoAgente)
    print(f"recursion_limit = {limite}:")
    for x in agente_con_guardia.invoke(entrada, {"recursion_limit": limite})["messages"][1:]:
        print("   ", linea(x))

Con $L = 5$, el modelo pide `integrar` en el superstep 3, con $\rho = 2$, y la guardia cambia ese mensaje por el aviso; sin ella, la ejecución terminaba en `GraphRecursionError` (Corolario 4.4). Con $L = 6$, el pedido llega con $\rho = 3$ y el resultado es el de la sección 3.

Las dos ejecuciones con el modelo insistente y $L = 7$, sin guardia y con ella:

<img src="../../imagenes/diagramas/modulo-1/pasos-restantes.svg" alt="Dos filas de supersteps con recursion_limit = 7 y el modelo insistente. Hasta el superstep 4 son iguales, con los pasos restantes bajando de 6 a 3. En el superstep 5, con ρ = 2, sin guardia el modelo pide herramientas, tools corre en el 6 y el modelo en el 7 con ρ = 0, y la ejecución termina en GraphRecursionError; con guardia, el modelo escribe el aviso sin pedidos y la ejecución llega a END en seis supersteps" width="640">

**Observación (`create_react_agent`).** El agente prearmado de `langgraph.prebuilt`, `create_react_agent`, trae esta guardia con la clave `remaining_steps`, pero con umbral 2: cambia el mensaje si $\rho_t < 2$ (función `_are_more_steps_needed`, en `langgraph/prebuilt/chat_agent_executor.py`). Con ese umbral, la Proposición 4.9 falla para algunos $L$; el ejercicio 3 pide encontrarlos. En langgraph 1.x, `create_react_agent` está marcado como obsoleto, en favor de `create_agent`, de `langchain.agents`.